# Notebook 04 - BERT y Aplicaciones

## Objetivos
- Clasificar sentimiento con DistilBERT fine-tuned.
- Responder preguntas con pipeline de QA.
- Extraer entidades (NER) y embeddings contextuales.

## Introduccion
BERT es encoder-only: excelente para entender texto. Exploraremos sentimiento, QA, NER y vectores `[CLS]` sobre datasets del curso.

In [4]:
# Escribe tu codigo aqui
from pathlib import Path
from IPython.display import display
import torch
import pandas as pd
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
)

RUTA_REVIEWS = Path("..")/'datasets'/'reviews_sentiment.csv'
RUTA_QA = Path("..")/'datasets'/'documentos_qa.csv'
print("Archivo listados: ", RUTA_REVIEWS.exists(), RUTA_QA.exists())


Archivo listados:  True True


## 1) Sentimiento con modelo fine-tuned SST-2

In [10]:
# Escribe tu codigo aqui
sentimientos = pipeline(
    'sentiment-analysis',
    model='pysentimiento/robertuito-sentiment-analysis'
)

df_reviews = pd.read_csv(RUTA_REVIEWS)
display(df_reviews.head(3))

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--pysentimiento--robertuito-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Fal

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

Device set to use cpu


,texto,sentimiento
0,"Excelente producto, llegó rápido y funciona pe...",positivo
1,"Muy mala calidad, se rompió al segundo día.",negativo
2,El servicio al cliente fue amable y resolvió m...,positivo


In [8]:
# Escribe tu codigo aqui
#traducir la texto
mapa_en = {
    "Excelente producto, llegó rápido y funciona perfectamente": "Excellent product, arrived quickly and works perfectly",
    "Muy mala calidad, se rompió al segundo día": "Very bad quality, broke on the second day",
    "El servicio al cliente fue amable y resolvió mi problema": "The customer service was friendly and resolved my problem"
}

result = []
for _, row in df_reviews.iterrows():
    review = row['texto']
    review_en = mapa_en.get(review, review)  # Traducir si está en el mapa, sino usar el original
    sentiment = sentimientos(review_en)[0]
    result.append({
        'review': review,
        'review_en': review_en,
        'label': sentiment['label'],
        'score': sentiment['score']
    })

df_result = pd.DataFrame(result)
display(df_result.head(3))

,review,review_en,label,score
0,"Excelente producto, llegó rápido y funciona pe...","Excelente producto, llegó rápido y funciona pe...",POSITIVE,0.996626
1,"Muy mala calidad, se rompió al segundo día.","Muy mala calidad, se rompió al segundo día.",NEGATIVE,0.943457
2,El servicio al cliente fue amable y resolvió m...,El servicio al cliente fue amable y resolvió m...,NEGATIVE,0.891604


In [9]:
nueva_frase_ingles = "The product is bad and did not meet my expectations"

sentimiento_nueva_frase = sentimientos(nueva_frase_ingles)[0]
print(f"Sentiment for the new phrase: {sentimiento_nueva_frase['label']} with score {sentimiento_nueva_frase['score']:.4f}")

Sentiment for the new phrase: NEGATIVE with score 0.9998


In [11]:
nueva_frase_espa = "El producto es malo y no cumplió mis expectativas"

sentimiento_nueva_frase = sentimientos(nueva_frase_espa)[0]
print(f"Sentiment for the new phrase: {sentimiento_nueva_frase['label']} with score {sentimiento_nueva_frase['score']:.4f}")

Sentiment for the new phrase: NEG with score 0.9720


## 2) Question Answering sobre documentos

In [22]:
# Escribe tu codigo aqui
qa = pipeline(
    'question-answering',
    model='mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es'
)
df_qa = pd.read_csv(RUTA_QA)
display(df_qa.head(3))

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--mrm8488--bert-base-spanish-wwm-cased-finetuned-spa-squad2-es. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is no

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Device set to use cpu


,contexto,pregunta,respuesta
0,Python fue creado por Guido van Rossum y publi...,¿Quién creó Python?,Guido van Rossum
1,Python fue creado por Guido van Rossum y publi...,¿En qué año se publicó Python?,1991
2,Los Transformers fueron introducidos en el pap...,¿Qué empresa introdujo los Transformers?,Google


In [20]:
# Escribe tu codigo aqui
respuestas = []

for _, row in df_qa.iterrows():
    out = qa(question=row['pregunta'], context=row['contexto'])
    respuestas.append({
        'pregunta': row['pregunta'],
        'respuestas_esperadas': row['respuesta'],
        'respuesta_obtenida': out['answer'],
        'score': out['score']
    })
display(pd.DataFrame(respuestas))

,pregunta,respuestas_esperadas,respuesta_obtenida,score
0,¿Quién creó Python?,Guido van Rossum,fue creado por Guido van Rossum y publicado en...,0.173339
1,¿En qué año se publicó Python?,1991,fue creado por Guido van Rossum y publicado en...,0.168586
2,¿Qué empresa introdujo los Transformers?,Google,fueron introducidos,0.080640
3,¿En qué año se publicó el paper?,2017,Los Transformers fueron introducidos,0.328462
4,¿Qué tipo de arquitectura usa BERT?,encoder-only,entrenado con Masked Language Modeling y Next ...,0.006823
5,¿Qué predice GPT?,la siguiente palabra,la siguiente,0.019914
6,¿Qué permite la Self-Attention?,que cada token observe a todos los demás tokens,que cada token,0.176362
7,¿Cómo reformula T5 las tareas?,texto a texto,T5 reformula todas las tareas de NLP,0.014026


In [23]:
# Escribe tu codigo aqui
respuestas = []

for _, row in df_qa.iterrows():
    out = qa(question=row['pregunta'], context=row['contexto'])
    respuestas.append({
        'pregunta': row['pregunta'],
        'respuestas_esperadas': row['respuesta'],
        'respuesta_obtenida': out['answer'],
        'score': out['score']
    })
display(pd.DataFrame(respuestas))

,pregunta,respuestas_esperadas,respuesta_obtenida,score
0,¿Quién creó Python?,Guido van Rossum,Guido van Rossum,0.968221
1,¿En qué año se publicó Python?,1991,1991,0.962721
2,¿Qué empresa introdujo los Transformers?,Google,Google,0.741518
3,¿En qué año se publicó el paper?,2017,2017,0.916031
4,¿Qué tipo de arquitectura usa BERT?,encoder-only,encoder-only,0.304605
5,¿Qué predice GPT?,la siguiente palabra,la siguiente palabra,0.623783
6,¿Qué permite la Self-Attention?,que cada token observe a todos los demás tokens,que cada token observe a todos los demás token...,0.175661
7,¿Cómo reformula T5 las tareas?,texto a texto,problemas de texto a texto,0.230271


## 3) Named Entity Recognition (NER)

In [ ]:
# Escribe tu codigo aqui


## 4) Extraccion de embeddings contextuales

In [ ]:
# Escribe tu codigo aqui


## Resultados
Clasificamos reseñas, respondimos preguntas extractivas, detectamos entidades y extrajimos un embedding `[CLS]` contextual.

## Conclusiones
BERT destaca en comprension y tareas de entendimiento. Para produccion conviene validar idioma, calibrar umbrales y considerar modelos multilingues.

## Ejercicios guiados resueltos
**Ejercicio:** Calcula accuracy aproximada mapeando POSITIVE->positivo.

**Solucion:**

In [ ]:
# Escribe tu codigo aqui


## Ejercicios propuestos
1. Prueba un modelo multilingue para reseñas en espanol.
2. Fine-tune ligero de sentimiento con `Trainer`.
3. Compara similitud coseno entre embeddings de dos frases.

## Preguntas de reflexion
1. Cuando falla QA extractivo aun con contexto correcto?
2. Que diferencia hay entre NER pipeline y token classification?
3. Por que `[CLS]` resume la secuencia en BERT?